# Lesson 28: Overfitting and Cross-Validation

**In class we'll run a few cells from Part I.** Part II (more features, with `sklearn` pipelines) is optional reading.

[Acknowledgments Page](https://ds100.org/sp24/acks/)

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
import plotly.express as px
import plotly.graph_objects as go
pd.options.mode.chained_assignment = None 
warnings.simplefilter(action='ignore', category=UserWarning)
np.random.seed(42)

In [ ]:
from sklearn.linear_model import LinearRegression

# Complexity and Overfitting 

For this notebook, we will use the seaborn `mpg` dataset which describes the fuel mileage (measured in miles per gallon or mpg) of various cars along with characteristics of those cars.  Our goal will be to build a model that can predict the fuel mileage of a car based on the characteristics of that car.

Consider the following `vehicles` dataset, which includes information about cars:

In [ ]:
vehicles = sns.load_dataset("mpg").dropna().rename(columns = {"horsepower": "hp"}).sort_values("hp")
vehicles.head()

Suppose we want to use the `hp` (horsepower) of a car to predict its `mpg` (gas mileage in miles per gallon). If we visualize the relationship between these two variables, we see a non-linear curvature. Fitting a linear model to these variables results in a high (poor) mean squared error (MSE). 

$$\hat{y} = \theta_0 + \theta_1 (\text{hp})$$

In [ ]:
X = vehicles[["hp"]]
y = vehicles["mpg"]

hp_model = LinearRegression()
hp_model.fit(X, y)
hp_model_predictions = hp_model.predict(X)

print(f"MSE of model with (hp) feature: {np.mean((y-hp_model_predictions)**2)}")

fig = px.scatter(vehicles, x="hp", y="mpg", width=800, height=600)
fig.add_trace(go.Scatter(x=vehicles["hp"], y=hp_model_predictions,
                         mode="lines", name="Linear Prediction"))
# sns.scatterplot(data=vehicles, x="hp", y="mpg")
# plt.plot(vehicles["hp"], hp_model_predictions, c="tab:red");

### Polynomial Features

To capture the non-linear relationship between the variables, we can introduce a non-linear feature: `hp` squared. Our new model is:

$$\hat{y} = \theta_0 + \theta_1 (\text{hp}) + \theta_2 (\text{hp}^2)$$

In [ ]:
X = vehicles[["hp"]]
X.loc[:, "hp^2"] = vehicles["hp"]**2

hp2_model = LinearRegression()
hp2_model.fit(X, y)
hp2_model_predictions = hp2_model.predict(X)


print(f"MSE of model with (hp^2) feature: {np.mean((y-hp2_model_predictions)**2)}")

fig = px.scatter(vehicles, x="hp", y="mpg", width=800, height=600)
fig.add_trace(go.Scatter(x=vehicles["hp"], y=hp2_model_predictions,
                         mode="lines", name="Quadratic Prediction"))
# sns.scatterplot(data=vehicles, x="hp", y="mpg")
# plt.plot(vehicles["hp"], hp2_model_predictions, c="tab:red");

What if we take things further and add even *more* polynomial features?

The cell below fits models of increasing complexity and computes their MSEs.

In [ ]:
def mse(predictions, observations):
    return np.mean((observations - predictions)**2)

# Add hp^3 and hp^4 as features to the data
X["hp^3"] = vehicles["hp"]**3
X["hp^4"] = vehicles["hp"]**4

# Fit a model with order 3
hp3_model = LinearRegression()
hp3_model.fit(X[["hp", "hp^2", "hp^3"]], vehicles["mpg"])
hp3_model_predictions = hp3_model.predict(X[["hp", "hp^2", "hp^3"]])

# Fit a model with order 4
hp4_model = LinearRegression()
hp4_model.fit(X[["hp", "hp^2", "hp^3", "hp^4"]], vehicles["mpg"])
hp4_model_predictions = hp4_model.predict(X[["hp", "hp^2", "hp^3", "hp^4"]])

In [ ]:
fig = px.scatter(vehicles, x="hp", y="mpg", width=800, height=600)
fig.add_trace(go.Scatter(
    x=vehicles["hp"], y=hp_model_predictions, mode="lines", 
    name=f"HP MSE={np.round(mse(vehicles['mpg'], hp_model_predictions),3)}"))
fig.add_trace(go.Scatter(
    x=vehicles["hp"], y=hp2_model_predictions, mode="lines", 
    name=f"HP^2 MSE={np.round(mse(vehicles['mpg'], hp2_model_predictions),3)}"))
fig.add_trace(go.Scatter(
    x=vehicles["hp"], y=hp3_model_predictions, mode="lines", 
    name=f"HP^3 MSE={np.round(mse(vehicles['mpg'], hp3_model_predictions),3)}"))
fig.add_trace(go.Scatter(
    x=vehicles["hp"], y=hp4_model_predictions, mode="lines", 
    name=f"HP^4 MSE={np.round(mse(vehicles['mpg'], hp4_model_predictions),3)}"))

Visualizing differently:

In [ ]:
# Plot the models' predictions
fig, ax = plt.subplots(1, 3, dpi=200, figsize=(12, 3))

predictions_dict = {0:hp2_model_predictions, 1:hp3_model_predictions, 2:hp4_model_predictions}

for i in predictions_dict:
    ax[i].scatter(vehicles["hp"], vehicles["mpg"], edgecolor="white", lw=0.5)
    ax[i].plot(vehicles["hp"], predictions_dict[i], "tab:red")
    ax[i].set_title(f"Model with order {i+2}")
    ax[i].set_xlabel("hp")
    ax[i].set_ylabel("mpg")
    ax[i].annotate(f"MSE: {np.round(mse(vehicles['mpg'], predictions_dict[i]), 3)}", (120, 40))

plt.subplots_adjust(wspace=0.3);

## Complexity and Overfitting

What we saw above was the phenomenon of **model complexity** – as we add additional features to the design matrix, the model becomes increasingly *complex*. Models with higher complexity have lower values of training error. Intuitively, this makes sense: with more features at its disposal, the model can match the observations in the training data more and more closely. 

We can run an experiment to see this in action. In the cell below, we fit many models of progressively higher complexity, then plot the MSE of predictions on the training set. 

The code used (specifically, the `Pipeline` and `PolynomialFeatures` functions of `sklearn`) is out of scope.

The **order** of a polynomial model is the highest power of any term in the model. An order 0 model takes the form $\hat{y} = \theta_0$, while an order 4 model takes the form $\hat{y} = \theta_0 + \theta_1 x + \theta_2 x^2 + \theta_3 x^3 + \theta_4 x^4$.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures

def fit_model_dataset(degree, dataset):
    pipelined_model = Pipeline([
            ('polynomial_transformation', PolynomialFeatures(degree)),
            ('linear_regression', LinearRegression())    
        ])

    pipelined_model.fit(dataset[["hp"]], dataset["mpg"])
    return mse(dataset['mpg'], pipelined_model.predict(dataset[["hp"]]))

errors = [fit_model_dataset(degree, vehicles) for degree in range(0, 8)]
MSEs_and_k = pd.DataFrame({"k": range(0, 8), "MSE": errors})

plt.plot(range(0, 8), errors)
plt.xlabel("Model Complexity (degree of polynomial)")
plt.ylabel("Training MSE");

def plot_degree_k_model(k, MSEs_and_k, axs):
    pipelined_model = Pipeline([
        ('poly_transform', PolynomialFeatures(degree = k)),
        ('regression', LinearRegression(fit_intercept = True))    
    ])
    pipelined_model.fit(vehicles[["hp"]], vehicles["mpg"])
    
    row = k // 4
    col = k % 4
    ax = axs[row, col]
    
    sns.scatterplot(data=vehicles, x='hp', y='mpg', ax=ax)
    
    x_range = np.linspace(45, 210, 100).reshape(-1, 1)
    ax.plot(x_range, pipelined_model.predict(pd.DataFrame(x_range, columns=['hp'])), c='tab:red', linewidth=2)
    
    ax.set_ylim((0, 50))
    mse_str = f"MSE: {MSEs_and_k.loc[k, 'MSE']:.4}\nDegree: {k}"
    ax.text(130, 35, mse_str, dict(size=14))

fig = plt.figure(figsize=(15, 6), dpi=150)
axs = fig.subplots(nrows=2, ncols=4)

for k in range(8):
    plot_degree_k_model(k, MSEs_and_k, axs)
fig.subplots_adjust(wspace=0.4, hspace=0.3)

As the model increases in polynomial degree (that is, it increases in complexity), the training MSE decreases, leveling off at roughly 18.

In fact, it is a mathematical fact that if we create a polynomial model with degree $n-1$, we can *perfectly* fit any $n$ points that have distinct $x$ values. For example, a set of 5 points can be perfectly modeled by a degree 4 model.

In [ ]:
np.random.seed(101)

fig, ax = plt.subplots(1, 3, dpi=200, figsize=(12, 3))

for i in range(0, 3):
    points = 3*np.random.uniform(size=(5, 2))

    polynomial_model = Pipeline([
                ('polynomial_transformation', PolynomialFeatures(4)),
                ('linear_regression', LinearRegression())    
            ])

    polynomial_model.fit(points[:, [0]], points[:, 1])

    ax[i].scatter(points[:, 0], points[:, 1])

    xs = np.linspace(0, 3)
    ax[i].plot(xs, polynomial_model.predict(xs[:, np.newaxis]), c="tab:red");

You may be tempted to always design models with high polynomial degree – after all, we know that we could theoretically achieve perfect predictions by creating a model with enough polynomial features. 

It turns out that the examples we looked at above represent a somewhat artificial scenario: we trained our model on all the data we had available, then used the model to make predictions on this very same dataset. A more realistic situation is when we wish to apply our model on unseen data – that is, datapoints that it did not encounter during the model fitting process. 

Suppose we obtain a random sample of 6 datapoints from our population of vehicle data. We want to train a model on these 6 points and use it to make predictions on unseen data (perhaps cars for which we don't already know the true `mpg`). 

In [ ]:
np.random.seed(100)

sample_6 = vehicles.sample(6)

sns.scatterplot(data=sample_6, x="hp", y="mpg")
plt.ylim(-35, 50)
plt.xlim(0, 250);

If we design a model with polynomial degree 5, we can make perfect predictions on this sample of training data.

In [ ]:
degree_5_model = Pipeline([
                ('polynomial_transformation', PolynomialFeatures(5)),
                ('linear_regression', LinearRegression())    
            ])

degree_5_model.fit(sample_6[["hp"]], sample_6["mpg"])
xs = np.linspace(0, 250, 1000)
degree_5_model_predictions = degree_5_model.predict(xs[:, np.newaxis])

plt.plot(xs, degree_5_model_predictions, c="tab:red")
sns.scatterplot(data=sample_6, x="hp", y="mpg", s=50)
plt.ylim(-35, 50)
plt.xlim(0, 250);

However, when we reapply this fitted model to the full population of data, it fails to capture the major trends of the dataset.

In [ ]:
plt.plot(xs, degree_5_model_predictions, c="tab:red")
sns.scatterplot(data=vehicles, x="hp", y="mpg", s=20)
plt.ylim(-35, 50);

The model has **overfit** to the data used to train it. It has essentially "memorized" the six datapoints used during model fitting, and does not generalize well to new data. 

Complex models tend to be more sensitive to the data used to train them. The **variance** of a model refers to its tendency to vary depending on the training data used during model fitting. It turns out that our degree-5 model has very high model variance. If we randomly sample new sets of datapoints to use in training, the model varies erratically. 

In [ ]:
np.random.seed(100)

fig, ax = plt.subplots(1, 3, dpi=200, figsize=(12, 3))

for i in range(0, 3):
    sample = vehicles.sample(6)

    polynomial_model = Pipeline([
                ('polynomial_transformation', PolynomialFeatures(5)),
                ('linear_regression', LinearRegression())    
            ])

    polynomial_model.fit(sample[["hp"]], sample["mpg"])

    ax[i].scatter(sample[["hp"]], sample["mpg"])

    xs = np.linspace(50, 210, 1000)
    ax[i].plot(xs, polynomial_model.predict(xs[:, np.newaxis]), c="tab:red")
    ax[i].set_ylim(-80, 100)
    ax[i].set_xlabel("hp")
    ax[i].set_ylabel("mpg")
    ax[i].set_title(f"Resample #{i+1}")
    
fig.tight_layout();

Visualizing differently:

In [ ]:
np.random.seed(16)

fig = px.scatter(vehicles, x="hp", y="mpg", width=800, height=600)

for i in range(0, 3):
    sample = vehicles.sample(6).sort_values("hp")
    
    polynomial_model = Pipeline([
                ('polynomial_transformation', PolynomialFeatures(5)),
                ('linear_regression', LinearRegression())    
            ])

    polynomial_model.fit(sample[["hp"]], sample["mpg"])
    
    name = f"Sample {i+1}"
    fig.add_trace(go.Scatter(x=sample["hp"], y=sample["mpg"], 
        name=name, legendgroup=name, mode="markers", showlegend=False,
        marker_size=16,
        marker_color=px.colors.qualitative.Plotly[i+1]))
    xs = np.linspace(10, 250, 1000)
    fig.add_trace(go.Scatter(x=xs, 
        y=polynomial_model.predict(xs[:, np.newaxis]),
        name=name, legendgroup=name, mode="lines",
        line_color=px.colors.qualitative.Plotly[i+1]))

fig.update_yaxes(range=[0, 50])

## Demo: Adding Columns of Random Numbers

In Lesson 27, the RMSE went down every time we added a feature. Here's an extreme test: add features that are pure **random noise**, with no relationship to `mpg` at all.

We'll fit each model on 75% of the cars and then check it on the other 25%, cars the model never saw while fitting.

In [ ]:
rng = np.random.default_rng(3022)

# 200 columns of random numbers: they have nothing to do with mpg
noise = pd.DataFrame(rng.normal(size=(len(vehicles), 200)), index=vehicles.index,
                     columns=[f"noise_{j}" for j in range(1, 201)])

# Fit on 75% of the cars; keep the other 25% aside as "new" cars
fit_cars = vehicles.sample(frac=0.75, random_state=3022)
new_cars = vehicles.drop(fit_cars.index)

results = []
for n_noise in [0, 1, 5, 20, 50, 100, 200]:
    features = ["hp"] + list(noise.columns[:n_noise])
    X_noise = pd.concat([vehicles[["hp"]], noise], axis=1)[features]

    model = LinearRegression().fit(X_noise.loc[fit_cars.index], fit_cars["mpg"])
    results.append({
        "Noise columns added": n_noise,
        "MSE on the cars we fit to": mse(model.predict(X_noise.loc[fit_cars.index]), fit_cars["mpg"]),
        "MSE on new cars": mse(model.predict(X_noise.loc[new_cars.index]), new_cars["mpg"]),
    })

noise_results = pd.DataFrame(results).set_index("Noise columns added")
noise_results.round(2)

In [ ]:
noise_results.plot(marker="o", figsize=(7, 4))
plt.xlabel("Number of random-noise columns added")
plt.ylabel("MSE")
plt.title("Random features: better fit, worse predictions");

Every column of noise **lowers** the MSE on the cars we fit to, but **raises** the MSE on new cars.

* Why can't the fit get worse? Least squares could always set a useless feature's coefficient to 0 and get back the old fit. So the MSE on the fitted data can only stay the same or go down.
* Why does it go down? By chance, random numbers always line up a little with the residuals, so least squares gives them small nonzero coefficients. That's the model fitting the **noise** in this particular sample, which doesn't carry over to new cars.

So a lower error on the data we fit to doesn't mean a better model. To choose between models, we need to measure error on data the model **wasn't** fit to. That's the rest of this lesson.

# Choosing the Polynomial Degree

Let's return to the `vehicles` dataset.

The polynomial degree is a **hyperparameter**: something we choose *before* fitting the model. Training error can't pick it (it always prefers the most complex model), so we'll estimate how each degree does on data it wasn't trained on.

Every model below is a small `sklearn` pipeline:

1. **Standardize `hp`** (subtract its mean, divide by its SD). Raw `hp` is about 100, so $\text{hp}^8 \approx 10^{16}$. Numbers that large make the computer's least squares solution unstable, and the errors would reflect rounding, not overfitting. Standardizing doesn't change which curves the model can fit.
2. Create the polynomial features $\text{hp}, \text{hp}^2, \dots, \text{hp}^d$.
3. Fit a linear regression.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import mean_squared_error

def poly_model(degree):
    """A degree-`degree` polynomial regression of mpg on hp."""
    return Pipeline([
        ("standardize", StandardScaler()),
        ("polynomial_features", PolynomialFeatures(degree)),
        ("linear_regression", LinearRegression()),
    ])

X = vehicles[["hp"]]
Y = vehicles["mpg"]

## Train-Test Split:
The first thing we will want to do with this data is construct a train/test split. Constructing a train/test split before EDA and data cleaning can often be helpful.  This allows us to see if our data cleaning and any conclusions we draw from visualizations generalize to new data. This can be done by re-running the data cleaning and EDA process on the test dataset.


We set the test set aside now and **don't touch it again until we've chosen our final model.**

### Using Pandas Operations

We can sample the entire dataset to get a permutation and then select a range of rows.

In [ ]:
shuffled_data = X.sample(frac=1., random_state=42) #we use random_state here so we all get the same permutation
shuffled_data

Selecting a range of rows for training and test

In [ ]:
split_point = int(shuffled_data.shape[0]*0.80)
split_point

In [ ]:
tr = shuffled_data.iloc[:split_point]
te = shuffled_data.iloc[split_point:]

In [ ]:
tr.head()

In [ ]:
te.head()

Checking that they add up.

In [ ]:
len(tr) + len(te) == len(X)

### Using SKLearn

We can use the `train_test_split` function from `sklearn.model_selection` to do this easily.

In [ ]:
from sklearn.model_selection import train_test_split

# `test_size` specifies the proportion of the full dataset that should be allocated to testing.
# `random_state` makes our results reproducible for educational purposes.
# shuffle is True by default and randomizes the data before splitting.

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, 
                                                    test_size=0.2, 
                                                    random_state=100, 
                                                    shuffle=True)

print(f"Size of full dataset: {X.shape[0]} points")
print(f"Size of training set: {X_train.shape[0]} points")
print(f"Size of test set: {X_test.shape[0]} points")

We fit models using only the training set. For example, a degree-2 model:

In [ ]:
model = poly_model(2)
model.fit(X_train, Y_train)

train_error = mean_squared_error(Y_train, model.predict(X_train))
print(f"Training MSE, degree 2: {train_error:.2f}")

We could compute this model's test error now, but if we used the test error to choose the degree, the test set would no longer be "unseen" data. Instead, we hold out part of the **training** set for that job.

## Validation Sets

To assess model performance on unseen data, then *use* this information to finetune the model, we introduce a validation set. You can imagine this as us splitting the training set into a validation set and a "mini" training set.

In [ ]:
# Split X_train further into X_train_mini and X_val.
X_train_mini, X_val, Y_train_mini, Y_val = train_test_split(X_train, Y_train, test_size=0.2, random_state=100)

print(f"Size of original training set: {X_train.shape[0]} points")
print(f"Size of mini training set: {X_train_mini.shape[0]} points")
print(f"Size of validation set: {X_val.shape[0]} points")

In the cell below, we repeat the experiment from above: fit several models of increasing complexity, then compute their errors. Here, we find the model's errors on the **validation set** to understand how model complexity influences performance on unseen data.

In [ ]:
fig, ax = plt.subplots(1, 3, dpi=200, figsize=(12, 3))
hp_grid = pd.DataFrame({"hp": np.linspace(45, 230, 200)})

for j, degree in enumerate([2, 3, 4]):
    model = poly_model(degree)
    model.fit(X_train_mini, Y_train_mini)
    val_mse = mean_squared_error(Y_val, model.predict(X_val))

    ax[j].scatter(X_val["hp"], Y_val, edgecolor="white", lw=0.5)
    ax[j].plot(hp_grid["hp"], model.predict(hp_grid), "tab:red")
    ax[j].set_title(f"Model with degree {degree}")
    ax[j].set_xlabel("hp")
    ax[j].set_ylabel("mpg")
    ax[j].annotate(f"Validation MSE: {val_mse:.3f}", (90, 40))

plt.subplots_adjust(wspace=0.3);

Let's repeat this for degrees 0 through 12:

1. Fit a degree-$d$ model to the mini training set.
2. Compute its MSE on the validation set.

We can use the validation set as many times as we like to *compare* models. We just never use it to *fit* them.

In [ ]:
degrees = range(0, 13)
val_errors = []
for degree in degrees:
    model = poly_model(degree).fit(X_train_mini, Y_train_mini)
    val_errors.append(mean_squared_error(Y_val, model.predict(X_val)))

holdout = pd.DataFrame({"Degree": degrees, "Validation MSE": val_errors}).set_index("Degree")

plt.figure(dpi=120)
plt.plot(degrees, val_errors, marker="o")
plt.xlabel("Model complexity (degree of polynomial)")
plt.ylabel("Validation MSE")
plt.xticks(degrees);

In [ ]:
holdout.round(2)

In [ ]:
print(f"Degree with the lowest validation MSE: {holdout['Validation MSE'].idxmin()}")

Validation MSE drops sharply from degree 0 to degree 2, then barely moves: every degree from 2 through 12 is within about 0.8 of the others. Degree 5 is technically the lowest (13.42), but degree 2 (13.87) is nearly as good. A difference that small could easily come from *which* 63 cars landed in the validation set. When several models score about the same, **prefer the simplest one.**

That's one validation set. Would a different split give a different answer?

## k-fold Cross-Validation

The validation set gave us an opportunity to understand how the model performs on a **single** set of unseen data. The specific validation set we drew was fixed – we used the same validation points every time.

It's possible that we may have, by random chance, selected a set of validation points that was *not* representative of other unseen data that the model might encounter (for example, if we happened to have selected all outlying data points for the validation set).

Different train/validation splits lead to different validation errors:

In [ ]:
# Try 3 different train/validation splits for the same degree-2 model
for i in range(1, 4):
    X_tr, X_va, Y_tr, Y_va = train_test_split(X_train, Y_train, test_size=0.2, random_state=i)
    model = poly_model(2).fit(X_tr, Y_tr)
    print(f"Validation MSE from split #{i}: {mean_squared_error(Y_va, model.predict(X_va)):.2f}")

The *same* model gets noticeably different validation errors depending on the split. **k-fold cross-validation** averages over $k$ splits so that no single lucky or unlucky validation set decides for us.

<br/><br/><br/>


---


## k-Fold Cross-Validation for One Model

We use `sklearn`'s `KFold` class. This creates 5 folds (shuffling the rows first):

```python
five_fold = KFold(n_splits=5, shuffle=True, random_state=25)
```

Then `five_fold.split(X_train)` gives the row positions for each round: `train_idx` (the 4 folds we fit on) and `valid_idx` (the 1 fold we validate on):

```python
for train_idx, valid_idx in five_fold.split(X_train):
```

Here is the 5-fold cross-validation MSE for a degree-2 model:

In [ ]:
from sklearn.model_selection import KFold

five_fold = KFold(n_splits=5, shuffle=True, random_state=25)
validation_errors = []

for train_idx, valid_idx in five_fold.split(X_train):
    # Split the training set into 4 folds for fitting and 1 fold for validation
    split_X_train, split_X_valid = X_train.iloc[train_idx], X_train.iloc[valid_idx]
    split_Y_train, split_Y_valid = Y_train.iloc[train_idx], Y_train.iloc[valid_idx]

    # Fit on the 4 folds, then compute the MSE on the held-out fold
    model = poly_model(2).fit(split_X_train, split_Y_train)
    validation_errors.append(mean_squared_error(split_Y_valid, model.predict(split_X_valid)))

print("Validation MSE for each fold:", np.round(validation_errors, 2))
print(f"5-fold cross-validation MSE, degree 2: {np.mean(validation_errors):.2f}")

## Using k-Fold Cross-Validation to Choose the Degree

Now we repeat this for every candidate degree, from 0 through 10, and choose the degree with the lowest cross-validation MSE. Every degree uses the **same** 5 folds, so they're compared on equal footing.

In [ ]:
def cv_mse(degree, folds=5, seed=25):
    """Average validation MSE of a degree-`degree` model over k folds of the training set."""
    kf = KFold(n_splits=folds, shuffle=True, random_state=seed)
    errors = []
    for train_idx, valid_idx in kf.split(X_train):
        model = poly_model(degree).fit(X_train.iloc[train_idx], Y_train.iloc[train_idx])
        errors.append(mean_squared_error(Y_train.iloc[valid_idx], model.predict(X_train.iloc[valid_idx])))
    return np.mean(errors)

cv_results = pd.DataFrame({"Degree": range(0, 11)})
cv_results["5-fold CV MSE"] = [cv_mse(d) for d in cv_results["Degree"]]
cv_results.set_index("Degree").round(2)

Plot the cross-validation MSE next to the training MSE (each model fit on the whole training set):

In [ ]:
cv_results["Training MSE"] = [mean_squared_error(Y_train, poly_model(d).fit(X_train, Y_train).predict(X_train)) for d in cv_results["Degree"]]

plot_df = cv_results[cv_results["Degree"] >= 1]   # leave off degree 0 so the scale shows the gap
plt.figure(dpi=120)
plt.plot(plot_df["Degree"], plot_df["Training MSE"], marker="o", label="Training MSE")
plt.plot(plot_df["Degree"], plot_df["5-fold CV MSE"], marker="o", label="5-fold CV MSE")
plt.xlabel("Model complexity (degree of polynomial)")
plt.ylabel("MSE")
plt.xticks(plot_df["Degree"])
plt.legend();

In [ ]:
best_degree = cv_results.loc[cv_results["5-fold CV MSE"].idxmin(), "Degree"]
print(f"Degree with the lowest 5-fold CV MSE: {best_degree}")

Training MSE keeps falling as the degree grows. The cross-validation MSE drops sharply to **degree 2**, changes little over the next few degrees, and climbs at high degrees, where the more complex models overfit.

How much should we trust the exact minimum? Let's reshuffle the folds (a different `random_state`) a few times and see which degree comes out lowest:

In [ ]:
for seed in [1, 2, 3, 4, 5]:
    errors = [cv_mse(d, seed=seed) for d in range(1, 8)]
    print(f"Folds shuffled with seed {seed}: lowest CV MSE at degree {1 + int(np.argmin(errors))}  "
          f"(degree 2: {errors[1]:.2f}, lowest: {min(errors):.2f})")

The "best" degree changes with the shuffle, but the differences are small: a few tenths, less than the wobble from reshuffling. A CV estimate is itself noisy. When the curve is that flat, take the **simplest model that's about as good: degree 2**.

Our final model:

$$\hat{y} = \theta_0 + \theta_1 (\text{hp}) + \theta_2 (\text{hp}^2)$$

(with hp standardized inside the pipeline).

*Note:* the CV Explorer demo from class uses a different random split (314 training and 78 test cars), so its numbers differ slightly from this notebook's, but the pattern and the choice are the same.

## Fit the Final Model on All the Training Data

Now that we've chosen the degree, we refit that model on the **entire** training set (all 5 folds together) to get the final $\theta$ values.

In [ ]:
final_degree = 2
final_model = poly_model(final_degree).fit(X_train, Y_train)

## Use the Test Set (Once) for the Final Result

We're done designing the model, so now (and only now) we evaluate it on the test set. This number estimates how the model will do on new cars. We don't go back and change the model based on it.

In [ ]:
plt.scatter(X_test["hp"], Y_test)
plt.plot(hp_grid["hp"], final_model.predict(hp_grid), color="red")
plt.xlabel("hp")
plt.ylabel("mpg")
plt.title(f"Final model (degree {final_degree}) with the test data")

print(f"Test MSE of the final degree-{final_degree} model: {mean_squared_error(Y_test, final_model.predict(X_test)):.2f}")

## Summary

* **Training error** always falls as a model gets more complex, so it can't tell us when to stop.
* A **validation set** (held out from the training set) estimates error on unseen data, so we can use it to choose hyperparameters like the polynomial degree.
* **k-fold cross-validation** averages the validation error over $k$ folds, so the choice doesn't hinge on one lucky or unlucky split.
* When several models score about the same, prefer the **simplest**.
* The **test set** is used once, at the very end, to report how the final model does.

<br/><br/>

---

# Part II (Optional): Adding More Features

*Optional reading, not covered in class.* This part uses `sklearn` pipelines to add more features and compares the models with 4-fold cross-validation.

Let's go through the process of adding more features to our model. 

We'll start with a model using hp up to hp^5 and then add engine characteristics like "cylinders" and the "displacement". 

For efficiency will use `sklearn` `Pipelines` to combine feature functions and models.  
For CSCI 3022, you **do not** need to know how to use `Pipeline`s in scikit-learn. They are quite involved. Fortunately, they are merely an accessory to the concepts of this lecture, and not the core of it. If you treat each instance of a `Pipeline` as a black-box way of specifying which features our model should have, you will be able to understand the cross-validation content just fine.




As we will see, by combining the feature function and model, we can simplify subsequent training and testing since we are guaranteed that our feature functions are the same on both the training and test datasets. 

In [ ]:
vehicles = sns.load_dataset("mpg").dropna().rename(columns = {"horsepower": "hp"}).sort_values("hp")
vehicles.head()

In [ ]:
from sklearn.model_selection import train_test_split

# `test_size` specifies the proportion of the full dataset that should be allocated to testing.
# `random_state` makes our results reproducible for educational purposes.
# shuffle is True by default and randomizes the data before splitting.


X = vehicles.drop(columns = ["mpg"])
Y= vehicles["mpg"]

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, 
                                                    test_size=0.2, 
                                                    random_state=100, 
                                                    shuffle=True)

print(f"Size of full dataset: {X.shape[0]} points")
print(f"Size of training set: {X_train.shape[0]} points")
print(f"Size of test set: {X_test.shape[0]} points")

In [ ]:
# Split X_train further into X_train_mini and X_val.
X_train_mini, X_val, Y_train_mini, Y_val = train_test_split(X_train, Y_train, test_size=0.2, random_state=100)

print(f"Size of original training set: {X_train.shape[0]} points")
print(f"Size of mini training set: {X_train_mini.shape[0]} points")
print(f"Size of validation set: {X_val.shape[0]} points")

In [ ]:
X_train_mini.head()

In [ ]:
def rmse(y, yhat):
    return np.sqrt(np.mean((y - yhat)**2))

<br/><br/><br/>

---

## SKLearn Pipelines

Again, for CSCI 3022, you do not need to know how to use `Pipeline`s in scikit-learn. They are quite involved. Fortunately, they are merely an accessory to the concepts of this lecture, and not the core of it. If you treat each instance of a `Pipeline` as a black-box way of specifying which features our model should have, you will be able to understand the cross-validation content just fine.

We have removed much of the dialogue around `Pipeline` from this lecture, but if you're interested, you can [skim the documentation on pipelines](https://scikit-learn.org/stable/modules/compose.html#pipeline).

Each pipeline below also has a `Scale` step (`StandardScaler`) just before the regression. Features like hp^5 are in the billions, and putting every feature on the same scale keeps the least squares computation numerically stable.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

model = Pipeline([
    ("SelectColumns", ColumnTransformer([("keep", "passthrough", ["hp"])])),
    ("Scale", StandardScaler()),   # put features on the same scale (hp^5 is huge)
    ("LinearModel", LinearRegression())
])

model.fit(X_train, Y_train);

#### Keeping track of all the models.

In this notebook (and in life) we will want to keep track of all our models.  Here I will store the models in a dictionary with a (not great) name so I can remember which model is which and can easily compare my models in a plot.

In [ ]:
models = {"hp": model}

### More Feature Transformations
As we saw above, we may want to use horsepower up to the 5th power.  We'll start with adding up to **hp^2**.  This is an additional feature transformation that we can add to the first stage of our pipeline.  To define this transformation we first need to create a function transformer:

In [ ]:
from sklearn.preprocessing import FunctionTransformer

def compute_square(X):
    return np.expand_dims(X[:,0]**2  , axis=1)

square_transformer = FunctionTransformer(compute_square, validate=True)

We can then add this as an additional column transformation:

In [ ]:
model = Pipeline([
    ("SelectColumns", ColumnTransformer([
        ("keep", "passthrough", ["hp"]),
        ("hp^2", square_transformer, ["hp"])])),
    ("Scale", StandardScaler()),   # put features on the same scale (hp^5 is huge)
    ("LinearModel", LinearRegression())
])

In [ ]:
model.fit(X_train, Y_train);

Again, we evaluate the model on our training dataset to see the error:

In [ ]:
Y_hat = model.predict(X_train)
Y = Y_train
print("Training Error (MSE):", mse(Y, Y_hat))

In [ ]:
models["hp+hp2"] = model

**Adding up to 5th degree of hp**

In [ ]:
def compute_cube(X):
    return np.expand_dims(X[:,0]**3  , axis=1)

cube_transformer = FunctionTransformer(compute_cube, validate=True)

In [ ]:
def compute_4th(X):
    return np.expand_dims(X[:,0]**4  , axis=1)

fourth_transformer = FunctionTransformer(compute_4th, validate=True)

In [ ]:
def compute_5th(X):
    return np.expand_dims(X[:,0]**5  , axis=1)

fifth_transformer = FunctionTransformer(compute_5th, validate=True)

In [ ]:
model = Pipeline([
    ("SelectColumns", ColumnTransformer([
        ("keep", "passthrough", ["hp"]),
        ("hp^2", square_transformer, ["hp"]),
        ("hp^3", cube_transformer, ["hp"]),
        ("hp^4", fourth_transformer, ["hp"]),
        ("hp^5", fifth_transformer, ["hp"])])),
    ("Scale", StandardScaler()),   # put features on the same scale (hp^5 is huge)
    ("LinearModel", LinearRegression())
])

In [ ]:
model.fit(X_train, Y_train);

In [ ]:
Y_hat = model.predict(X_train)
Y = Y_train
print("Training Error (MSE):", mse(Y, Y_hat))

In [ ]:
models["hp..hp5"] = model

### Adding More Features

We might also want to add cylinders and displacement.   

We can then add these as additional column transformation:

In [ ]:
model = Pipeline([
    ("SelectColumns", ColumnTransformer([("keep", "passthrough", ["cylinders", "displacement", "hp"]),
        ("hp^2", square_transformer, ["hp"]),
        ("hp^3", cube_transformer, ["hp"]),
        ("hp^4", fourth_transformer, ["hp"]),
        ("hp^5", fifth_transformer, ["hp"])])),
    ("Scale", StandardScaler()),   # put features on the same scale (hp^5 is huge)
    ("LinearModel", LinearRegression())
])

In [ ]:
model.fit(X_train, Y_train);

Again, we evaluate the model on our training dataset. Adding cylinders and displacement lowers the training error:

In [ ]:
Y_hat = model.predict(X_train)
Y = Y_train
print("Training Error (MSE):", mse(Y, Y_hat))

In [ ]:
models["hp..hp5+c+d"] = model

### More Feature Transformations

We might also want to look at the **displacement per cylinder**.  This is an additional feature transformation that we can add to the first stage of our pipeline.  To define this transformation we first need to create a function transformer:

In [ ]:
def compute_volume(X):
    return np.expand_dims(X[:,1] / X[:,0]  , axis=1)

volume_transformer = FunctionTransformer(compute_volume, validate=True)

In [ ]:
model = Pipeline([
    ("SelectColumns", ColumnTransformer([
        ("keep", "passthrough", ["cylinders", "displacement", "hp"]),
        ("hp^2", square_transformer, ["hp"]),
        ("hp^3", cube_transformer, ["hp"]),
        ("hp^4", fourth_transformer, ["hp"]),
        ("hp^5", fifth_transformer, ["hp"]),
        ("cyl_vol", volume_transformer, ["cylinders", "displacement"])])),
    ("Scale", StandardScaler()),   # put features on the same scale (hp^5 is huge)
    ("LinearModel", LinearRegression())
])

In [ ]:
model.fit(X_train, Y_train);

Again, we evaluate the model on our training dataset and see a reduction in error:

In [ ]:
Y_hat = model.predict(X_train)
Y = Y_train
print("Training Error (MSE):", mse(Y, Y_hat))

In [ ]:
models["hp..hp5+c+d+d/c"] = model

### Adding More Features

We can now add additional features about the car.

In [ ]:
quantitative_features = ["cylinders", "displacement","hp" ,"weight", "acceleration"]
model = Pipeline([
    ("SelectColumns", ColumnTransformer([
        ("keep", "passthrough", quantitative_features),
        ("hp^2", square_transformer, ["hp"]),
        ("hp^3", cube_transformer, ["hp"]),
        ("hp^4", fourth_transformer, ["hp"]),
        ("hp^5", fifth_transformer, ["hp"]),
        ("cyl_vol", volume_transformer, ["cylinders", "displacement"])])),
    ("Scale", StandardScaler()),   # put features on the same scale (hp^5 is huge)
    ("LinearModel", LinearRegression())
])

## Dealing with NaN data

If there are NaN (missing values) in the data we will need to deal with these missing values.  In previous lectures I mentioned imputation and a standard imputation technique is to replace the missing value with the mean for that column.  Scikit learn has a built-in imputation function that we can add to our pipeline after we select the desired columns.  The imputation will actually be applied to all the columns.  If we wanted to apply it to a specific column then we would need to put it inside the ColumnTransformer.

**Notice:** The imputation function actually needs to be **fit** to data so it is also part of the model.

In [ ]:
from sklearn.impute import SimpleImputer
model = Pipeline([
    ("SelectColumns", ColumnTransformer([
        ("keep", "passthrough", quantitative_features),
        ("hp^2", square_transformer, ["hp"]),
        ("hp^3", cube_transformer, ["hp"]),
        ("hp^4", fourth_transformer, ["hp"]),
        ("hp^5", fifth_transformer, ["hp"]),
        ("cyl_vol", volume_transformer, ["cylinders", "displacement"])])),
    ("Imputation", SimpleImputer()),
    ("Scale", StandardScaler()),   # put features on the same scale (hp^5 is huge)
    ("LinearModel", LinearRegression())
])

We can now train our model.

In [ ]:
quantitative_features = ["cylinders", "displacement", "hp", "weight", "acceleration"]

In [ ]:
model.fit(X_train, Y_train);

Saving the model for later comparison:

In [ ]:
models['hp..hp5+c+d+d/c+w+a'] = model

Evaluating the training error:

In [ ]:
Y_hat = model.predict(X_train)
Y = Y_train
print("Training Error (MSE):", mse(Y, Y_hat))

We reduced the training error but what about the test error?  We really shouldn't look at the test error so instead we will use cross validation to compare the accuracy:

<br/><br/><br/>

---

## Cross Validation

In the following function we use the sklearn `KFold` cross validation class. 

Here we define a four fold cross validation with 

```python 
four_fold = KFold(n_splits=4)
```

Then we loop over the 4 splits and get the indices (`tr_ind`) in the training data to use for training and the indices (`va_ind`) in the training data to use for validation:

```python
for tr_ind, va_ind in four_fold.split(X_train):
```

In [ ]:
from sklearn.model_selection import KFold
from sklearn.base import clone

def cross_validate_mse(model):
    model = clone(model)
    four_fold = KFold(n_splits=4)
    mse_values = []
    for tr_ind, va_ind in four_fold.split(X_train):
        model.fit(X_train.iloc[tr_ind,:], Y_train.iloc[tr_ind])
        mse_values.append(mse(Y_train.iloc[va_ind], model.predict(X_train.iloc[va_ind,:])))
    return np.mean(mse_values)

Validating the model:

In [ ]:
cross_validate_mse(model)

The following helper function generates a plot comparing all the models in the models dictionary.

In [ ]:
import plotly.offline as py
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.figure_factory as ff

In [ ]:
def compare_models(models):
    # Compute the training error for each model
    training_mse = [mse(Y_train, model.predict(X_train)) for model in models.values()]
    # Compute the cross validation error for each model
    validation_mse = [cross_validate_mse(model) for model in models.values()]
    # Compute the test error for each model (don't do this!)
    test_mse = [mse(Y_test, model.predict(X_test)) for model in models.values()]
    names = list(models.keys())
    fig = go.Figure([
        go.Bar(x = names, y = training_mse, name="Training MSE"),
        go.Bar(x = names, y = validation_mse, name="CV MSE"),
        go.Bar(x = names, y = test_mse, name="Test MSE", opacity=.3)])
    return fig

In [ ]:
list(models.keys())

In [ ]:
fig = compare_models(models)
fig.update_yaxes(range=[2,28], title="MSE")

Notice I made the **Test MSE** invisible(ish) because you shouldn't look at it until we are done.  But again for demonstration purposes I plotted it so we can see how it compares to the training and cross validation errors.

Can you improve the model further?  Let's try adding the model year

In [ ]:
quantitative_features = ["cylinders", "displacement", "hp", "weight", "acceleration", "model_year"]
model = Pipeline([
    ("SelectColumns", ColumnTransformer([
        ("keep", "passthrough", quantitative_features),
        ("hp^2", square_transformer, ["hp"]),
        ("hp^3", cube_transformer, ["hp"]),
        ("hp^4", fourth_transformer, ["hp"]),
        ("hp^5", fifth_transformer, ["hp"]),
        ("cyl_vol", volume_transformer, ["cylinders", "displacement"])])),
    ("Imputation", SimpleImputer()),
    ("Scale", StandardScaler()),   # put features on the same scale (hp^5 is huge)
    ("LinearModel", LinearRegression())
])

In [ ]:
model.fit(X_train, Y_train)
models['h..hp^5+c+d+d/c+w+a+y'] = model

Comparing the models

In [ ]:
fig = compare_models(models)
fig.update_yaxes(range=[2,28], title="MSE")

The model year improved accuracy quite a bit!  This improvement also appears to generalize as it also reduced the cross validation error. 

## Going too Far?

Can we use the **car's name** to predict MPG?  The name contains general features like the manufacturer that might help but it also contains the vehicle make which is probably too specific and not all the vehicles in test will appear in training. 

Let's try by applying the CountVectorizer (which implements something called a "Bag of Words"  (https://www.mygreatlearning.com/blog/bag-of-words/).  At this point we are also likely to have too many dimensions in our model and we are not applying any regularization technique to compensate (because we haven't covered regularization in lecture yet.)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
model = Pipeline([
    ("SelectColumns", ColumnTransformer([
        ("keep", "passthrough", quantitative_features),
        ("cyl_vol", volume_transformer, ["cylinders", "displacement"]),
        ("text", CountVectorizer(), "name")
    ])),
    ("Imputation", SimpleImputer()),
    ("Scale", StandardScaler(with_mean=False)),   # put features on the same scale (hp^5 is huge)
    ("LinearModel", LinearRegression())
])

**Notice**: That we are using an additional column transformation.

In [ ]:
model.fit(X_train, Y_train)
models['hp..hp^5+c+d+d/c+w+a+y+n'] = model

In [ ]:
fig = compare_models(models)
fig.update_yaxes(range=[0,28], title="MSE")

**Overfitting!**.  We substantially reduced the training error but actually made the generalization error worse!

## Finalizing the Model:

In [ ]:
best_model = clone(models['h..hp^5+c+d+d/c+w+a+y'])

In [ ]:
best_model.fit(X_train, Y_train);

In [ ]:
mse(best_model.predict(X_test), Y_test)

## Limitations of Using Cross Validation:
Using cross-validation to manage model complexity has a couple of critical limitations: typically it requires the complexity to vary discretely; and there may not be a natural way to order the models. Rather than changing the dimensions of a sequence of models, we can fit a large model and apply constraints on the size of the coefficients. This notion is called regularization and is the topic of the next section.